<a href="https://colab.research.google.com/github/HowardChen0211/dataManPy/blob/main/ProblemSet_1_pandas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introduction

## U.S. Housing Market Trends

### Research Question

- How have home values, rental prices, and housing inventory changed across major U.S. metropolitan areas, and to what extent is home-value growth associated with rent growth?

- Which cities have seen the fastest rising housing prices?

- Correlation between home price increases and rent increases?


### Unit of Analysis

One observation = one U.S. metropolitan area in one month.



### Objectives

1. Clean and standardize Zillow housing datasets.
2. Transform monthly housing data into an analysis-ready format.
3. Integrate home value, rental, and inventory datasets.
4. Analyze housing trends across metropolitan areas.
5. Examine the relationship between home value and rent growth.

# ProblemSet 0

## Import Libraries

In [169]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

#will display all output not just last command
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

## Load the data

In [170]:
# Zillow Home Value Index (ZHVI)
!wget -q -O ZHVI.csv https://files.zillowstatic.com/research/public_csvs/zhvi/Metro_zhvi_uc_sfrcondo_tier_0.33_0.67_sm_sa_month.csv?t=1788564553
# Zillow Observed Rent Index (ZORI)
!wget -q -O ZORI.csv https://files.zillowstatic.com/research/public_csvs/zori/Metro_zori_uc_sfrcondomfr_sm_month.csv?t=1788564553
# For-Sale Inventory
!wget -q -O inventory.csv https://files.zillowstatic.com/research/public_csvs/invt_fs/Metro_invt_fs_uc_sfrcondo_sm_month.csv?t=1788564553

In [171]:
!ls

inventory.csv  sample_data  ZHVI.csv  ZORI.csv


In [172]:
# Read the data
zhvi = pd.read_csv("ZHVI.csv")
zori = pd.read_csv("ZORI.csv")
inve = pd.read_csv("inventory.csv")

In [173]:
# See how many features in the data
print("Home Index: ", zhvi.columns)
print("Rent Index: ", zori.columns)
print("House inventory: ", inve.columns)

Home Index:  Index(['RegionID', 'SizeRank', 'RegionName', 'RegionType', 'StateName',
       '2000-01-31', '2000-02-29', '2000-03-31', '2000-04-30', '2000-05-31',
       ...
       '2025-11-30', '2025-12-31', '2026-01-31', '2026-02-28', '2026-03-31',
       '2026-04-30', '2026-05-31', '2026-06-30', '2026-07-31', '2026-08-31'],
      dtype='object', length=325)
Rent Index:  Index(['RegionID', 'SizeRank', 'RegionName', 'RegionType', 'StateName',
       '2015-01-31', '2015-02-28', '2015-03-31', '2015-04-30', '2015-05-31',
       ...
       '2025-11-30', '2025-12-31', '2026-01-31', '2026-02-28', '2026-03-31',
       '2026-04-30', '2026-05-31', '2026-06-30', '2026-07-31', '2026-08-31'],
      dtype='object', length=145)
House inventory:  Index(['RegionID', 'SizeRank', 'RegionName', 'RegionType', 'StateName',
       '2018-03-31', '2018-04-30', '2018-05-31', '2018-06-30', '2018-07-31',
       ...
       '2025-11-30', '2025-12-31', '2026-01-31', '2026-02-28', '2026-03-31',
       '2026-04-30', 

In [174]:
# take a look about the data (first 5 rows)
print(zhvi.head())

   RegionID  SizeRank       RegionName RegionType StateName  2000-01-31  \
0    102001         0    United States    country       NaN  123577.760   
1    394913         1     New York, NY        msa        NY  220013.124   
2    753899         2  Los Angeles, CA        msa        CA  221912.270   
3    394463         3      Chicago, IL        msa        IL  156387.254   
4    394514         4       Dallas, TX        msa        TX  129720.858   

   2000-02-29  2000-03-31  2000-04-30  2000-05-31  ...  2025-11-30  \
0  123795.172  124064.796  124643.087  125308.630  ...  365789.637   
1  220948.055  221891.646  223803.726  225783.751  ...  708946.176   
2  222738.065  223838.016  226026.550  228420.120  ...  948164.084   
3  156532.023  156807.738  157493.649  158318.717  ...  344798.361   
4  129778.636  129845.058  130017.548  130244.973  ...  367550.988   

   2025-12-31  2026-01-31  2026-02-28  2026-03-31  2026-04-30  2026-05-31  \
0  366588.387  367350.447  368071.714  368567.173  

In [175]:
# take a look about the data (first 5 rows)
print(zori.head())

   RegionID  SizeRank       RegionName RegionType StateName  2015-01-31  \
0    102001         0    United States    country       NaN    1140.638   
1    394913         1     New York, NY        msa        NY    2269.050   
2    753899         2  Los Angeles, CA        msa        CA    1750.925   
3    394463         3      Chicago, IL        msa        IL    1355.484   
4    394514         4       Dallas, TX        msa        TX    1050.959   

   2015-02-28  2015-03-31  2015-04-30  2015-05-31  ...  2025-11-30  \
0    1146.954    1155.528    1164.297    1172.928  ...    1894.180   
1    2284.254    2303.730    2322.888    2338.066  ...    3438.814   
2    1761.763    1777.093    1792.357    1807.616  ...    2890.828   
3    1363.029    1373.212    1382.703    1392.164  ...    2099.210   
4    1055.772    1063.449    1074.602    1083.761  ...    1637.215   

   2025-12-31  2026-01-31  2026-02-28  2026-03-31  2026-04-30  2026-05-31  \
0    1891.578    1893.656    1900.768    1911.280  

In [176]:
# take a look about the data (first 5 rows)
print(inve.head())

   RegionID  SizeRank       RegionName RegionType StateName  2018-03-31  \
0    102001         0    United States    country       NaN 1421529.000   
1    394913         1     New York, NY        msa        NY   73707.000   
2    753899         2  Los Angeles, CA        msa        CA   21998.000   
3    394463         3      Chicago, IL        msa        IL   38581.000   
4    394514         4       Dallas, TX        msa        TX   24042.000   

   2018-04-30  2018-05-31  2018-06-30  2018-07-31  ...  2025-11-30  \
0 1500195.000 1592417.000 1660618.000 1709146.000  ... 1322714.000   
1   80345.000   85864.000   90067.000   91881.000  ...   45625.000   
2   23784.000   25605.000   27109.000   28811.000  ...   24235.000   
3   42253.000   45757.000   47492.000   48984.000  ...   22985.000   
4   25876.000   28224.000   30490.000   32408.000  ...   35783.000   

   2025-12-31  2026-01-31  2026-02-28  2026-03-31  2026-04-30  2026-05-31  \
0 1239487.000 1158753.000 1115629.000 1155751.000 1

In [177]:
# See how big the data
print(zhvi.shape) # (895, 324)
print(zori.shape) # (752, 144)
print(inve.shape) # (928, 106)

(895, 325)
(750, 145)
(928, 107)


In [178]:
# Descriptive Statistics
print(zhvi.describe())

        RegionID  SizeRank  2000-01-31  2000-02-29  2000-03-31  2000-04-30  \
count    895.000   895.000     431.000     432.000     433.000     435.000   
mean  412099.673   461.752  111209.790  111389.267  111547.705  112237.640   
std    78377.355   268.711   46309.247   46421.951   46611.996   47269.545   
min   102001.000     0.000   48084.016   48187.274   48279.020   48446.067   
25%   394546.000   230.500   80317.169   80449.475   80564.892   81185.854   
50%   394795.000   460.000   99795.493   99995.970   99862.205  100057.127   
75%   395044.500   689.500  127821.018  128030.928  128250.278  128371.986   
max   753929.000   939.000  357664.002  359507.371  362284.373  369804.693   

       2000-05-31  2000-06-30  2000-07-31  2000-08-31  ...  2025-11-30  \
count     437.000     438.000     439.000     440.000  ...     895.000   
mean   112972.627  113542.039  114329.901  114958.678  ...  295062.799   
std     47856.810   48475.317   49198.346   49933.929  ...  174008.557   
m

In [179]:
# prints information about a DataFrame
print(zhvi.info())
print(zori.info())
print(inve.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 895 entries, 0 to 894
Columns: 325 entries, RegionID to 2026-08-31
dtypes: float64(320), int64(2), object(3)
memory usage: 2.2+ MB
None
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 750 entries, 0 to 749
Columns: 145 entries, RegionID to 2026-08-31
dtypes: float64(140), int64(2), object(3)
memory usage: 849.7+ KB
None
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 928 entries, 0 to 927
Columns: 107 entries, RegionID to 2026-08-31
dtypes: float64(102), int64(2), object(3)
memory usage: 775.9+ KB
None


In [180]:
# Missing values
print(zhvi.isna().sum())
print(zori.isna().sum())
print(inve.isna().sum())

RegionID      0
SizeRank      0
RegionName    0
RegionType    0
StateName     1
             ..
2026-04-30    0
2026-05-31    0
2026-06-30    0
2026-07-31    0
2026-08-31    0
Length: 325, dtype: int64
RegionID       0
SizeRank       0
RegionName     0
RegionType     0
StateName      1
              ..
2026-04-30    83
2026-05-31    67
2026-06-30    58
2026-07-31    37
2026-08-31     0
Length: 145, dtype: int64
RegionID      0
SizeRank      0
RegionName    0
RegionType    0
StateName     1
             ..
2026-04-30    0
2026-05-31    0
2026-06-30    0
2026-07-31    0
2026-08-31    1
Length: 107, dtype: int64


In [181]:
# Check Duplicates
print(zhvi.duplicated().sum())
print(zori.duplicated().sum())
print(inve.duplicated().sum())

0
0
0


#### Make sure the data match across all three datasets, so I filtered the dates to include only records from 2020 to 2025.

In [182]:
# # ZHVI
# columns = ["RegionID", "SizeRank", "RegionName", "RegionType", "StateName"]

# filter_20 = columns + [i for i in zhvi.columns if i[0].isdigit() and i >= "2020-01"]

# zhvi_20 = zhvi[filter_20]

# zhvi_20

In [183]:
# # ZORI
# columns = ["RegionID", "SizeRank", "RegionName", "RegionType", "StateName"]

# filter_20 = columns + [i for i in zori.columns if i[0].isdigit() and i >= "2020-01"]

# zori_20 = zori[filter_20]

# zori_20

In [184]:
# # inventory
# columns = ["RegionID", "SizeRank", "RegionName", "RegionType", "StateName"]

# filter_20 = columns + [i for i in inve.columns if i[0].isdigit() and i >= "2020-01"]

# inve_20 = inve[filter_20]

# inve_20

#### Write a function for above code, so that only need to call once.

In [185]:
# I'm trying to write above code into a function, which makes it clean.

def cleaning_data(data, start = "2020-01", end = "2026-01"):

  columns = ["RegionID", "SizeRank", "RegionName", "RegionType", "StateName"]

  filter_20 = columns + [i for i in data.columns if i[0].isdigit() and i >= start and i < end]

  return data[filter_20]

zhvi, zori, inve = (cleaning_data(d) for d in (zhvi, zori, inve))

# ProblemSet 1

### Select Top 20 Metropolitan Areas as an example




In [186]:
# Location of the 20 largest MSAs in the United States, based on U.S. Census Bureau estimates for July 1, 2025
# Source: https://en.wikipedia.org/wiki/Metropolitan_statistical_area

metros = [
    "New York, NY",
    "Los Angeles, CA",
    "Chicago, IL",
    "Dallas, TX",
    "Houston, TX",
    "Atlanta, GA",
    "Washington, DC",
    "Miami, FL",
    "Philadelphia, PA",
    "Phoenix, AZ",
    "Boston, MA",
    "Riverside, CA",
    "San Francisco, CA",
    "Detroit, MI",
    "Seattle, WA",
    "Minneapolis, MN",
    "Tampa, FL",
    "San Diego, CA",
    "Denver, CO",
    "Orlando, FL"
]

zhvi = zhvi[zhvi["RegionName"].isin(metros)]

zori = zori[zori["RegionName"].isin(metros)]

inve = inve[inve["RegionName"].isin(metros)]

### ZHVI Data

In [187]:
# Make a copy so that don't alter the original data
zhvi = zhvi.copy()

In [188]:
zhvi.head()

,RegionID,SizeRank,RegionName,RegionType,StateName,2020-01-31,2020-02-29,2020-03-31,2020-04-30,2020-05-31,...,2025-03-31,2025-04-30,2025-05-31,2025-06-30,2025-07-31,2025-08-31,2025-09-30,2025-10-31,2025-11-30,2025-12-31
1,394913,1,"New York, NY",msa,NY,509099.781,511374.806,513991.525,516370.700,518256.333,...,694103.098,696558.135,698341.705,699829.652,701011.986,701808.587,703220.879,705575.941,708946.176,712394.824
2,753899,2,"Los Angeles, CA",msa,CA,669786.059,671322.675,672091.796,673060.035,672117.018,...,958497.027,954173.426,949393.042,945032.337,942436.260,941363.303,942421.673,944841.591,948164.084,952130.048
3,394463,3,"Chicago, IL",msa,IL,243955.285,245329.310,246770.912,247671.531,247767.314,...,337385.802,337998.297,338294.476,338662.352,339517.770,340532.629,341891.233,343221.050,344798.361,346563.957
4,394514,4,"Dallas, TX",msa,TX,263767.597,265348.907,266852.535,267644.058,267634.550,...,379285.098,377123.225,374667.587,372264.908,370215.279,368830.913,368173.259,367818.964,367550.988,367329.712
5,394692,5,"Houston, TX",msa,TX,230309.726,231179.260,232011.962,232528.166,232526.595,...,315444.366,314509.035,313549.910,312546.255,311694.430,310969.904,310494.433,310087.521,309895.060,309913.973


#### Clean the data

In [189]:
# See the data types in the dataframe
zhvi.dtypes

,0
RegionID,int64
SizeRank,int64
RegionName,object
RegionType,object
StateName,object
...,...
2025-08-31,float64
2025-09-30,float64
2025-10-31,float64
2025-11-30,float64


In [190]:
## Reshape the Data https://pandas.pydata.org/docs/reference/api/pandas.melt.html
## The data is often provided in a wide format, so I will unpivot it from wide to long format.

id_col = ["RegionID", "SizeRank", "RegionName", "RegionType", "StateName"]

date_col = []
for col in zhvi.columns:
  if col not in id_col:
    date_col.append(col)

zhvi = zhvi.melt(
    id_vars=id_col,
    value_vars=date_col,
    var_name="Date",
    value_name="ZHVI"
)

# Drop unused columns
zhvi = zhvi.drop(columns=["SizeRank", "RegionType"], axis=1)

In [191]:
# See the data types in the dataframe
zhvi.dtypes

,0
RegionID,int64
RegionName,object
StateName,object
Date,object
ZHVI,float64


In [192]:
zhvi.select_dtypes(include='float')

,ZHVI
0,509099.781
1,669786.059
2,243955.285
3,263767.597
4,230309.726
...,...
1435,386905.913
1436,927602.863
1437,360102.926
1438,571734.018


In [193]:
# Convert Dates to datetime type
zhvi["Date"] = pd.to_datetime(zhvi["Date"], errors = "coerce") # Turns invalid dates into NaT(Not a Time)

# Check missing value in Date column
zhvi["Date"].isna().sum()

np.int64(0)

In [194]:
# sort the data
zhvi = zhvi.sort_values(
    by = ["RegionName", "Date"]).reset_index(drop=True)

zhvi.head()

,RegionID,RegionName,StateName,Date,ZHVI
0,394347,"Atlanta, GA",GA,2020-01-31,252897.123
1,394347,"Atlanta, GA",GA,2020-02-29,254771.778
2,394347,"Atlanta, GA",GA,2020-03-31,256456.363
3,394347,"Atlanta, GA",GA,2020-04-30,257628.262
4,394347,"Atlanta, GA",GA,2020-05-31,258025.546


### ZORI Data

In [195]:
# Make a copy so that don't alter the original data
zori = zori.copy()

#### Clean the data

In [196]:
# See the data types in the dataframe
zori.dtypes

,0
RegionID,int64
SizeRank,int64
RegionName,object
RegionType,object
StateName,object
...,...
2025-08-31,float64
2025-09-30,float64
2025-10-31,float64
2025-11-30,float64


In [197]:
## Reshape the Data
## The data is often provided in a wide format, so I will unpivot it from wide to long format.

id_col = ["RegionID", "SizeRank", "RegionName", "RegionType", "StateName"]

date_columns = [
    col for col in zori.columns
    if col not in id_col
]

zori = zori.melt(
    id_vars = id_col,
    value_vars = date_columns,
    var_name = "Date",
    value_name = "ZORI"
)

# Drop unused columns
zori = zori.drop(columns=["SizeRank", "RegionType"], axis=1)

In [198]:
zori.dtypes

,0
RegionID,int64
RegionName,object
StateName,object
Date,object
ZORI,float64


In [199]:
zori.select_dtypes(include='float')

,ZORI
0,2583.127
1,2266.695
2,1548.867
3,1294.811
4,1338.697
...,...
1435,1663.555
1436,2912.131
1437,1979.495
1438,1869.106


In [200]:
# Convert Dates to datetime type
zori["Date"] = pd.to_datetime(
    zori["Date"],
    errors="coerce"
)

In [201]:
# sort the data
zori = zori.sort_values(
    by = ["RegionName", "Date"]).reset_index(drop=True)

zori.head()

,RegionID,RegionName,StateName,Date,ZORI
0,394347,"Atlanta, GA",GA,2020-01-31,1317.829
1,394347,"Atlanta, GA",GA,2020-02-29,1325.205
2,394347,"Atlanta, GA",GA,2020-03-31,1333.529
3,394347,"Atlanta, GA",GA,2020-04-30,1338.258
4,394347,"Atlanta, GA",GA,2020-05-31,1343.470


### Housing Inventory Data

In [202]:
# Make a copy so that don't alter the original data
inve = inve.copy()

#### Clean the data

In [203]:
# See the data types in the dataframe
inve.dtypes

,0
RegionID,int64
SizeRank,int64
RegionName,object
RegionType,object
StateName,object
...,...
2025-08-31,float64
2025-09-30,float64
2025-10-31,float64
2025-11-30,float64


In [204]:
## Reshape the Data
## The data is often provided in a wide format, so I will unpivot it from wide to long format.

id_col = ["RegionID", "SizeRank", "RegionName", "RegionType", "StateName"]

date_columns = [
    col for col in inve.columns
    if col not in id_col
]

inve = inve.melt(
    id_vars = id_col,
    value_vars = date_columns,
    var_name = "Date",
    value_name = "Inventory"
)

# Drop unused columns
inve = inve.drop(columns=["SizeRank", "RegionType"], axis=1)

In [205]:
inve.dtypes

,0
RegionID,int64
RegionName,object
StateName,object
Date,object
Inventory,float64


In [206]:
inve.select_dtypes(include='float')

,Inventory
0,74301.000
1,21649.000
2,38966.000
3,28058.000
4,27537.000
...,...
1435,9771.000
1436,6741.000
1437,21787.000
1438,12685.000


In [207]:
# Convert Dates to datetime type
inve["Date"] = pd.to_datetime(
    inve["Date"],
    errors="coerce"
)

In [208]:
# Sort the data
inve = inve.sort_values(
    by = ["RegionName", "Date"]).reset_index(drop=True)

inve.head()

,RegionID,RegionName,StateName,Date,Inventory
0,394347,"Atlanta, GA",GA,2020-01-31,28473.000
1,394347,"Atlanta, GA",GA,2020-02-29,27623.000
2,394347,"Atlanta, GA",GA,2020-03-31,28398.000
3,394347,"Atlanta, GA",GA,2020-04-30,28681.000
4,394347,"Atlanta, GA",GA,2020-05-31,29794.000


### Local Area Unemployment Statistics Data (Only 2020 ~ 2024)

Source:
- Bureau of Labor Statistics https://www.bls.gov/lau/metrossa.htm,
- Wikkipedia https://en.wikipedia.org/wiki/Metropolitan_statistical_area

(n) = No data available. (x) = Data unavailable due to the 2025 lapse in appropriations.

In [209]:
laborforce = pd.read_excel("https://github.com/HowardChen0211/dataManPy/raw/refs/heads/main/data/ssamatab1.xlsx", dtype={'ST FIPS Code': str, 'Area FIPS Code': str, 'Month': str})

In [210]:
laborforce.head()

,ST FIPS Code,Area FIPS Code,Area,Year,Month,Civilian Labor Force,Employment,Unemployment,Unemployment Rate
0,01,11500,"Anniston-Oxford, AL MSA",2020,01,50272,48493,1779,3.500
1,01,12220,"Auburn-Opelika, AL MSA",2020,01,91078,88405,2673,2.900
2,01,13820,"Birmingham, AL MSA",2020,01,555452,539409,16043,2.900
3,01,19300,"Daphne-Fairhope-Foley, AL MSA",2020,01,107204,104512,2692,2.500
4,01,19460,"Decatur, AL MSA",2020,01,71423,69384,2039,2.900


In [211]:
# Filter the Year only from 2020 ~ 2024
laborforce = laborforce[(laborforce["Year"] >= 2020) & (laborforce["Year"] <= 2024)]

In [212]:
# Select Top 20 MSA rank by population as of July 1, 2025
metros_area = [
    "New York-Newark-Jersey City, NY-NJ MSA",
    "Los Angeles-Long Beach-Anaheim, CA MSA",
    "Chicago-Naperville-Elgin, IL-IN MSA",
    "Dallas-Fort Worth-Arlington, TX MSA",
    "Houston-Pasadena-The Woodlands, TX MSA",
    "Atlanta-Sandy Springs-Roswell, GA MSA",
    "Washington-Arlington-Alexandria, DC-VA-MD-WV MSA",
    "Miami-Fort Lauderdale-West Palm Beach, FL MSA",
    "Philadelphia-Camden-Wilmington, PA-NJ-DE-MD MSA",
    "Phoenix-Mesa-Chandler, AZ MSA",
    "Boston-Cambridge-Newton, MA-NH MSA",
    "Riverside-San Bernardino-Ontario, CA MSA",
    "San Francisco-Oakland-Fremont, CA MSA",
    "Detroit-Warren-Dearborn, MI MSA",
    "Seattle-Tacoma-Bellevue, WA MSA",
    "Minneapolis-St. Paul-Bloomington, MN-WI MSA",
    "Tampa-St. Petersburg-Clearwater, FL MSA",
    "San Diego-Chula Vista-Carlsbad, CA MSA",
    "Denver-Aurora-Centennial, CO MSA",
    "Orlando-Kissimmee-Sanford, FL MSA"
]

laborforces = laborforce[laborforce["Area"].isin(metros_area)].reset_index(drop=True)

In [213]:
laborforces.head()

,ST FIPS Code,Area FIPS Code,Area,Year,Month,Civilian Labor Force,Employment,Unemployment,Unemployment Rate
0,04,38060,"Phoenix-Mesa-Chandler, AZ MSA",2020,01,2473431,2375478,97953,4
1,06,31080,"Los Angeles-Long Beach-Anaheim, CA MSA",2020,01,6901427,6609059,292368,4.200
2,06,40140,"Riverside-San Bernardino-Ontario, CA MSA",2020,01,2078931,1997238,81693,3.900
3,06,41740,"San Diego-Chula Vista-Carlsbad, CA MSA",2020,01,1619898,1569978,49920,3.100
4,06,41860,"San Francisco-Oakland-Fremont, CA MSA",2020,01,2587507,2520522,66985,2.600


In [214]:
laborforces.tail()

,ST FIPS Code,Area FIPS Code,Area,Year,Month,Civilian Labor Force,Employment,Unemployment,Unemployment Rate
1195,36,35620,"New York-Newark-Jersey City, NY-NJ MSA",2024,12,10320015,9863560,456455,4.400
1196,42,37980,"Philadelphia-Camden-Wilmington, PA-NJ-DE-MD MSA",2024,12,3287026,3149177,137849,4.200
1197,48,19100,"Dallas-Fort Worth-Arlington, TX MSA",2024,12,4527410,4350025,177385,3.900
1198,48,26420,"Houston-Pasadena-The Woodlands, TX MSA",2024,12,3882268,3711129,171139,4.400
1199,53,42660,"Seattle-Tacoma-Bellevue, WA MSA",2024,12,2325042,2226522,98520,4.200


In [215]:
laborforces.shape

(1200, 9)

In [216]:
# missing values
laborforces.isna().sum()

,0
ST FIPS Code,0
Area FIPS Code,0
Area,0
Year,0
Month,0
Civilian Labor Force,0
Employment,0
Unemployment,0
Unemployment Rate,0


In [217]:
# Check duplicated values
laborforces.duplicated().sum()

np.int64(0)

#### Clean the data

In [218]:
laborforces.dtypes

,0
ST FIPS Code,object
Area FIPS Code,object
Area,object
Year,int64
Month,object
Civilian Labor Force,object
Employment,object
Unemployment,object
Unemployment Rate,object


In [219]:
# rename the column
laborforces.rename(columns = {"ST FIPS Code": "st_fips",
    "Area FIPS Code": "cbsa_code",
    "Area": "area",
    "Year": "year",
    "Month": "month",
    "Civilian Labor Force": "labor_force",
    "Employment": "employment",
    "Unemployment": "unemployment",
    "Unemployment Rate": "unemp_rate"}, inplace = True)

# Drop unused columns
laborforces.drop(columns = ["st_fips",	"cbsa_code"], inplace = True)

# change the (n) / (x) to NaN
num_cols = ["labor_force", "employment", "unemployment", "unemp_rate"]
laborforces[num_cols] = laborforces[num_cols].apply(pd.to_numeric, errors="coerce")

# Convert the Data into datetime object
laborforces["date"] = pd.to_datetime(
    dict(year=laborforces["year"], month=laborforces["month"], day=1)
)

In [220]:
laborforces.head()

,area,year,month,labor_force,employment,unemployment,unemp_rate,date
0,"Phoenix-Mesa-Chandler, AZ MSA",2020,01,2473431,2375478,97953,4.000,2020-01-01
1,"Los Angeles-Long Beach-Anaheim, CA MSA",2020,01,6901427,6609059,292368,4.200,2020-01-01
2,"Riverside-San Bernardino-Ontario, CA MSA",2020,01,2078931,1997238,81693,3.900,2020-01-01
3,"San Diego-Chula Vista-Carlsbad, CA MSA",2020,01,1619898,1569978,49920,3.100,2020-01-01
4,"San Francisco-Oakland-Fremont, CA MSA",2020,01,2587507,2520522,66985,2.600,2020-01-01


In [221]:
laborforces.tail()

,area,year,month,labor_force,employment,unemployment,unemp_rate,date
1195,"New York-Newark-Jersey City, NY-NJ MSA",2024,12,10320015,9863560,456455,4.400,2024-12-01
1196,"Philadelphia-Camden-Wilmington, PA-NJ-DE-MD MSA",2024,12,3287026,3149177,137849,4.200,2024-12-01
1197,"Dallas-Fort Worth-Arlington, TX MSA",2024,12,4527410,4350025,177385,3.900,2024-12-01
1198,"Houston-Pasadena-The Woodlands, TX MSA",2024,12,3882268,3711129,171139,4.400,2024-12-01
1199,"Seattle-Tacoma-Bellevue, WA MSA",2024,12,2325042,2226522,98520,4.200,2024-12-01


In [222]:
laborforces.groupby(["area", "year"])[["unemp_rate"]].mean()

unemp_rate
area                                             year            
Atlanta-Sandy Springs-Roswell, GA MSA            2020       6.875
                                                 2021       3.933
                                                 2022       3.008
                                                 2023       3.158
                                                 2024       3.358
...                                                           ...
Washington-Arlington-Alexandria, DC-VA-MD-WV MSA 2020       6.392
                                                 2021       4.458
                                                 2022       2.792
                                                 2023       2.558
                                                 2024       3.067

[100 rows x 1 columns]

In [223]:
# Let's say want to see New York-Newark-Jersey City, NY-NJ MSA unemp rate within 5 years
laborforces.groupby(["area", "year"])[["unemp_rate"]].mean().loc["New York-Newark-Jersey City, NY-NJ MSA"]

,unemp_rate
year,
2020,10.400
2021,7.667
2022,4.400
2023,4.283
2024,4.458


In [224]:
# Let's say want to see Los Angeles–Long Beach–Anaheim, CA MSA unemp rate within 5 years
laborforces.groupby(["area", "year"])[["unemp_rate"]].mean().loc["Los Angeles-Long Beach-Anaheim, CA MSA"]

,unemp_rate
year,
2020,11.517
2021,8.275
2022,4.525
2023,4.683
2024,5.300


In [225]:
# Let's say want to see Philadelphia–Camden–Wilmington, PA-NJ-DE-MD MSA unemp rate within 5 years
laborforces.groupby(["area", "year"])[["unemp_rate"]].mean().loc["Philadelphia-Camden-Wilmington, PA-NJ-DE-MD MSA"]

,unemp_rate
year,
2020,8.892
2021,6.033
2022,3.983
2023,3.758
2024,4.000


We can see that the profound impact of the pandemic on the labor market in 2020.

In [226]:
laborforce_2020 = laborforces[laborforces["year"] == 2020]
laborforce_2021 = laborforces[laborforces["year"] == 2021]
laborforce_2022 = laborforces[laborforces["year"] == 2022]
laborforce_2023 = laborforces[laborforces["year"] == 2023]
laborforce_2024 = laborforces[laborforces["year"] == 2024]

In [227]:
pd.set_option('display.float_format', '{:.3f}'.format) # Turn off the scientific notation https://medium.com/@amit25173/steps-to-pandas-turn-off-scientific-notation-62c44b86ee24

#### Year in 2020 Laborforce data

In [228]:
# Just want to see 2020/04, which is the peak of the COVID-19
april = laborforce_2020.loc[laborforce_2020['month'] == '04', ['area', 'labor_force', 'unemp_rate']]

In [245]:
april.sort_values(by = 'unemp_rate', ascending = False)

,area,labor_force,unemp_rate
73,"Detroit-Warren-Dearborn, MI MSA",1933668,23.400
71,"Chicago-Naperville-Elgin, IL-IN MSA",4723993,19.000
79,"Seattle-Tacoma-Bellevue, WA MSA",2142469,18.200
72,"Boston-Cambridge-Newton, MA-NH MSA",2556268,16.600
61,"Los Angeles-Long Beach-Anaheim, CA MSA",6352402,16.500
68,"Orlando-Kissimmee-Sanford, FL MSA",1303606,16.200
63,"San Diego-Chula Vista-Carlsbad, CA MSA",1560261,16.100
62,"Riverside-San Bernardino-Ontario, CA MSA",2039760,15.900
75,"New York-Newark-Jersey City, NY-NJ MSA",9112349,14.600
76,"Philadelphia-Camden-Wilmington, PA-NJ-DE-MD MSA",3079426,14.300


In [230]:
# Filter Calfornia MSA
ca = laborforce_2020[laborforce_2020['area'].str.contains(r',\s*CA\b')] # matches the , and spaces zero or more whitespace

ca.head()

,area,year,month,labor_force,employment,unemployment,unemp_rate,date
1,"Los Angeles-Long Beach-Anaheim, CA MSA",2020,01,6901427,6609059,292368,4.200,2020-01-01
2,"Riverside-San Bernardino-Ontario, CA MSA",2020,01,2078931,1997238,81693,3.900,2020-01-01
3,"San Diego-Chula Vista-Carlsbad, CA MSA",2020,01,1619898,1569978,49920,3.100,2020-01-01
4,"San Francisco-Oakland-Fremont, CA MSA",2020,01,2587507,2520522,66985,2.600,2020-01-01
21,"Los Angeles-Long Beach-Anaheim, CA MSA",2020,02,6917193,6624355,292838,4.200,2020-02-01


In [231]:
#  NJ's MSA（NY-NJ、PA-NJ-DE-MD)
nj = laborforce_2020[laborforce_2020['area'].str.contains(r'\bNJ\b')] # set the word boundary

nj.head()

,area,year,month,labor_force,employment,unemployment,unemp_rate,date
15,"New York-Newark-Jersey City, NY-NJ MSA",2020,01,10128034,9748608,379426,3.700,2020-01-01
16,"Philadelphia-Camden-Wilmington, PA-NJ-DE-MD MSA",2020,01,3226585,3097194,129391,4.000,2020-01-01
35,"New York-Newark-Jersey City, NY-NJ MSA",2020,02,10137815,9757428,380387,3.800,2020-02-01
36,"Philadelphia-Camden-Wilmington, PA-NJ-DE-MD MSA",2020,02,3224590,3096702,127888,4.000,2020-02-01
55,"New York-Newark-Jersey City, NY-NJ MSA",2020,03,10062853,9567675,495178,4.900,2020-03-01


In [239]:
# Source:  https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.filter.html

laborforce_2020.filter(like='employ', axis = 1) ## did the same as laborforce_2020[["employment", "unemployment"]]

,employment,unemployment
0,2375478,97953
1,6609059,292368
2,1997238,81693
3,1569978,49920
4,2520522,66985
...,...,...
235,8813251,875423
236,2950702,247514
237,3773976,247224
238,3224267,267253


In [251]:
# Average unemployment rate, highest unemployment rate, and average labor force for each MSA
df_area = (laborforce_2020.groupby('area')
             .agg(avg_unemp=('unemp_rate', 'mean'),
                  max_unemp=('unemp_rate', 'max'),
                  avg_laborforce=('labor_force', 'mean'))
             .sort_values('max_unemp', ascending=False))
df_area

,avg_unemp,max_unemp,avg_laborforce
area,,,
"Detroit-Warren-Dearborn, MI MSA",11.633,23.400,2113538.167
"Orlando-Kissimmee-Sanford, FL MSA",10.442,21.900,1353910.083
"Chicago-Naperville-Elgin, IL-IN MSA",9.808,19.000,4729431.750
"Los Angeles-Long Beach-Anaheim, CA MSA",11.517,18.200,6546302.167
"Seattle-Tacoma-Bellevue, WA MSA",8.875,18.200,2159345.667
"New York-Newark-Jersey City, NY-NJ MSA",10.400,17.700,9716369.000
"Boston-Cambridge-Newton, MA-NH MSA",8.808,16.600,2720170.167
"San Diego-Chula Vista-Carlsbad, CA MSA",9.258,16.100,1569520.000
"Riverside-San Bernardino-Ontario, CA MSA",9.883,15.900,2073328.833


In [257]:
laborforce_2020.head()

,area,year,month,labor_force,employment,unemployment,unemp_rate,date
0,"Phoenix-Mesa-Chandler, AZ MSA",2020,01,2473431,2375478,97953,4.000,2020-01-01
1,"Los Angeles-Long Beach-Anaheim, CA MSA",2020,01,6901427,6609059,292368,4.200,2020-01-01
2,"Riverside-San Bernardino-Ontario, CA MSA",2020,01,2078931,1997238,81693,3.900,2020-01-01
3,"San Diego-Chula Vista-Carlsbad, CA MSA",2020,01,1619898,1569978,49920,3.100,2020-01-01
4,"San Francisco-Oakland-Fremont, CA MSA",2020,01,2587507,2520522,66985,2.600,2020-01-01


In [258]:
# The laborforce monthly trend in 2020

monthly_2020 = laborforce_2020.groupby(["area",'month'])[['labor_force', 'employment', 'unemployment']].sum()
monthly_2020['unemp_rate'] = monthly_2020['unemployment'] / monthly_2020['labor_force'] * 100

In [265]:
monthly_2020.loc["New York-Newark-Jersey City, NY-NJ MSA"]

,labor_force,employment,unemployment,unemp_rate
month,,,,
01,10128034,9748608,379426,3.746
02,10137815,9757428,380387,3.752
03,10062853,9567675,495178,4.921
04,9112349,7783500,1328849,14.583
05,9457925,7781502,1676423,17.725
06,9619356,8199023,1420333,14.765
07,9707144,8298077,1409067,14.516
08,9754575,8531611,1222964,12.537
09,9625166,8633217,991949,10.306


In [267]:
# Labor force changes from the beginning to the end of the year for each MSA in 2020
lf_change = (laborforce_2020.sort_values('date')
               .groupby('area')['labor_force']
               .agg(['first', 'last'])) # get the first & last in the group

lf_change['pct_change'] = (lf_change['last'] / lf_change['first'] - 1) * 100

In [268]:
lf_change

,first,last,pct_change
area,,,
"Atlanta-Sandy Springs-Roswell, GA MSA",3154291,3132236,-0.699
"Boston-Cambridge-Newton, MA-NH MSA",2793599,2714849,-2.819
"Chicago-Naperville-Elgin, IL-IN MSA",4877042,4681565,-4.008
"Dallas-Fort Worth-Arlington, TX MSA",3971347,4021200,1.255
"Denver-Aurora-Centennial, CO MSA",1694994,1676629,-1.083
"Detroit-Warren-Dearborn, MI MSA",2169406,2058650,-5.105
"Houston-Pasadena-The Woodlands, TX MSA",3513309,3491520,-0.620
"Los Angeles-Long Beach-Anaheim, CA MSA",6901427,6446249,-6.595
"Miami-Fort Lauderdale-West Palm Beach, FL MSA",3164171,2999847,-5.193


### Wikipedia Metropolitan statistical area population data

Source: https://en.wikipedia.org/wiki/Metropolitan_statistical_area

In [236]:
import requests


url = "https://en.wikipedia.org/wiki/Metropolitan_statistical_area"

header = {
  "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/152.0.0.0 Safari/537.36",
  "X-Requested-With": "XMLHttpRequest"
}

r = requests.get(url, headers=header, timeout = 10)

print(r)
# population = pd.read_html(r.text, match='Metropolitan') # scrape all HTML tables (the <table> tag) from webpage

<Response [403]>


In [237]:
population.head()

NameError: name 'population' is not defined

#### Clean the data

In [ ]:
population.rename(columns={
    "Metropolitan statistical area": "msa",
    "2025 estimate": "pop_2025",
    "2020 census[a]": "pop_2020"}, inplace = True)
population.drop(columns=["Encompassing combined statistical area"], inplace=True)

In [ ]:
population.dtypes

In [ ]:
population["% change"] = (
    population["% change"]
    .str.replace("%", "", regex=False)
    .str.replace("−", "-", regex=False)
    .astype(float)
)

In [ ]:
population

#### Cite: Below is using Claude for help

In [ ]:
# import io
# import re
# import unicodedata

# # Bureau of Labor Statistics using the Old name，but Wikipedia using 2023 OMB New name
# # https://www.google.com/search?q=Virginia+Beach%E2%80%93Norfolk%E2%80%93Newport+News%2C+VA-NC%2C+MSA+change+name%3F&rlz=1C5CHFA_enTW996TW996&oq=Virginia+Beach%E2%80%93Norfolk%E2%80%93Newport+News%2C+VA-NC%2C+MSA+change+name%3F&gs_lcrp=EgZjaHJvbWUyBggAEEUYOTIHCAEQIRigATIHCAIQIRiPAjIHCAMQIRiPAtIBCDg4MzNqMGo3qAIAsAIA&sourceid=chrome&source=chrome.ob&ie=UTF-8

# ALIASES = {
#     "virginia beach-chesapeake-norfolk, va-nc": "virginia beach-norfolk-newport news, va-nc",
#     "san juan-carolina-caguas, pr": "san juan-bayamon-caguas, pr",
# }

# def msa_key(s: str) -> str:
#     s = unicodedata.normalize("NFKD", str(s))
#     s = "".join(c for c in s if not unicodedata.combining(c))  # Mayagüez -> Mayaguez
#     s = re.sub(r"[\u2010-\u2015\u2212/]", "-", s)              # – — ／ change to  -
#     s = s.replace("\u2019", "'")
#     s = re.sub(r"\s+", " ", s).strip()
#     s = re.sub(r",?\s*MSA$", "", s, flags=re.I)
#     s = s.replace(".", "")                                     # St. -> St
#     s = re.sub(r"\s*-+\s*", "-", s)                            # A--B -> A-B
#     s = re.sub(r"\s*,\s*", ", ", s)
#     s = s.lower().strip()
#     return ALIASES.get(s, s)


# laborforce["msa_key"] = laborforce["area"].map(msa_key)
# population["msa_key"] = population["msa"].map(msa_key)

In [ ]:
# population.head()

In [ ]:
# # Collapse the monthly into annual figures, one row per (MSA, year).
# # Each MSA contributes 12 monthly observations per year; we need a single
# # annual value so the series can be joined to the static Census population data.

# laborforce = (laborforce.groupby(["msa_key", "year"])
#     .agg(area=("area", "last"), # "last" picks the most recent label
#          labor_force=("labor_force", "mean"),
#          employment=("employment", "mean"),
#          unemployment=("unemployment", "mean"))
#     .reset_index())

# laborforce["unemp_rate"] = laborforce["unemployment"] / laborforce["labor_force"] * 100

# # Excluding Puerto Rico
# population = population[~population["msa_key"].str.endswith(", pr")]
# laborforce = laborforce[~laborforce["msa_key"].str.endswith(", pr")]


# # is good
# # chk = population.merge(laborforce, on="msa_key", how="outer", indicator=True)
# # print(chk["_merge"].value_counts())
# # print(chk.loc[chk["_merge"] != "both", "msa_key"].unique())


# df = population.merge(
#     laborforce,
#     on="msa_key", how="inner", validate="one_to_many"
# )
# df

## Merge the ZHVI/ ZORI/ Invetory dataframe into One Datasets

In [ ]:
housing1 = zhvi.merge(
    zori,
    on=["RegionName", "StateName", "Date"],
    how="inner"
).drop(columns=["RegionID_y"])

In [ ]:
housing1.head()

In [ ]:
housing = housing1.merge(
    inve,
    on=["RegionName", "StateName", "Date"],
    how="inner"
).drop(columns=["RegionID"])

housing.rename(columns={"RegionID_x": "RegionID"}, inplace=True)

In [ ]:
housing.head()

In [ ]:
housing.info()

In [ ]:
housing.isna().sum()

In [ ]:
housing.duplicated(
    subset=["RegionName", "Date"]
).sum()

In [ ]:
housing

## Reference

1. Metropolitan statistical area: https://en.wikipedia.org/wiki/Metropolitan_statistical_area

2. Zillow Housing: https://www.zillow.com/research/data/

3. Source: U.S. Census Bureau, Population Division ;
Annual Estimates of the Resident Population for Metropolitan Statistical Areas in the United States and Puerto Rico: April 1, 2020 to July 1, 2025 (CBSA-MET-EST2025-POP)
